# ADA Lab Experiment 1: Algorithmic Strategies in Real-World Problem Solving

This notebook presents five problems based on major algorithmic paradigms discussed in Analysis and Design of Algorithms. Each section includes the problem context, implementation, complexity discussion, and short observations.

In [ ]:
import random
import time
import heapq
from pathlib import Path
import matplotlib.pyplot as plt

random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## Problem 1: Divide and Conquer - Merge Sort

### Problem Context
Sorting is a common requirement in data processing systems, search engines, and transaction analysis. Merge Sort applies the divide and conquer strategy by recursively splitting the input into smaller subproblems and then merging the sorted results.

### Why This Strategy Fits
The array can be divided into independent halves, solved recursively, and combined efficiently. This gives Merge Sort a predictable `O(n log n)` time complexity.

In [ ]:
def merge(left, right):
    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged

def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

sample_array = [38, 27, 43, 3, 9, 82, 10]
sorted_array = merge_sort(sample_array)

print('Input Array :', sample_array)
print('Sorted Array:', sorted_array)
print('Time Complexity  : O(n log n)')
print('Space Complexity : O(n)')
print('Observation      : Merge Sort remains efficient even when input size grows.')

## Problem 2: Sorting Performance Comparison - Merge Sort vs Bubble Sort

### Problem Context
Organizations often need to choose between simple and efficient sorting techniques. This comparison shows how execution time changes as the input size increases.

### Why This Comparison Matters
Bubble Sort is easy to understand but inefficient for large datasets, while Merge Sort is more scalable. The experiment below measures their execution time on the same random inputs.

In [ ]:
def bubble_sort(arr):
    result = arr.copy()
    n = len(result)

    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            if result[j] > result[j + 1]:
                result[j], result[j + 1] = result[j + 1], result[j]
                swapped = True
        if not swapped:
            break

    return result

sizes = [100, 300, 500, 700, 900]
bubble_times = []
merge_times = []

for size in sizes:
    data = [random.randint(1, 10000) for _ in range(size)]

    start = time.perf_counter()
    bubble_sort(data)
    bubble_times.append(time.perf_counter() - start)

    start = time.perf_counter()
    merge_sort(data)
    merge_times.append(time.perf_counter() - start)

print('Input Sizes       :', sizes)
print('Bubble Sort Times :', [round(value, 6) for value in bubble_times])
print('Merge Sort Times  :', [round(value, 6) for value in merge_times])

plt.figure(figsize=(8, 5))
plt.plot(sizes, bubble_times, marker='o', linewidth=2, label='Bubble Sort')
plt.plot(sizes, merge_times, marker='s', linewidth=2, label='Merge Sort')
plt.xlabel('Input Size')
plt.ylabel('Execution Time (seconds)')
plt.title('Sorting Performance Comparison')
plt.legend()
plt.tight_layout()
images_dir = Path('images')
images_dir.mkdir(exist_ok=True)
plot_path = images_dir / 'sorting_performance_comparison.png'
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()

print('Saved Plot        :', plot_path)
print('Observation: Bubble Sort grows much faster than Merge Sort as input size increases.')

## Problem 3: Greedy Strategy - Fractional Knapsack

### Problem Context
A business may need to allocate limited carrying capacity to items with different profits and weights. In the fractional version, items can be divided, so the goal is to maximize total value.

### Why This Strategy Fits
The greedy choice of selecting the highest value-to-weight ratio first leads to the optimal solution when fractions of items are allowed.

In [ ]:
def fractional_knapsack(values, weights, capacity):
    items = []
    for index, (value, weight) in enumerate(zip(values, weights), start=1):
        items.append((value / weight, value, weight, index))

    items.sort(reverse=True)

    total_value = 0.0
    selected_items = []

    for ratio, value, weight, index in items:
        if capacity == 0:
            break

        if weight <= capacity:
            total_value += value
            capacity -= weight
            selected_items.append((index, 1.0, value))
        else:
            fraction = capacity / weight
            gained_value = value * fraction
            total_value += gained_value
            selected_items.append((index, round(fraction, 2), round(gained_value, 2)))
            capacity = 0

    return round(total_value, 2), selected_items

values = [60, 100, 120]
weights = [10, 20, 30]
capacity = 50

max_value, chosen_items = fractional_knapsack(values, weights, capacity)
print('Values   :', values)
print('Weights  :', weights)
print('Capacity :', capacity)
print('Chosen Items (item, fraction, gained value):', chosen_items)
print('Maximum Value:', max_value)
print('Time Complexity  : O(n log n) due to sorting')
print('Space Complexity : O(n)')
print('Observation      : Greedy selection works because item fractions are allowed.')

## Problem 4: Dynamic Programming - 0/1 Knapsack

### Problem Context
In many real situations, items cannot be divided. A company must either take an item completely or leave it. The challenge is to maximize value without exceeding capacity.

### Why This Strategy Fits
The problem has overlapping subproblems and optimal substructure, which makes Dynamic Programming an appropriate approach.

In [ ]:
def zero_one_knapsack(capacity, weights, values):
    n = len(values)
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]

    for i in range(1, n + 1):
        for current_capacity in range(1, capacity + 1):
            if weights[i - 1] <= current_capacity:
                dp[i][current_capacity] = max(
                    values[i - 1] + dp[i - 1][current_capacity - weights[i - 1]],
                    dp[i - 1][current_capacity]
                )
            else:
                dp[i][current_capacity] = dp[i - 1][current_capacity]

    return dp[n][capacity], dp

capacity = 50
weights = [10, 20, 30]
values = [60, 100, 120]

best_value, dp_table = zero_one_knapsack(capacity, weights, values)
print('Values   :', values)
print('Weights  :', weights)
print('Capacity :', capacity)
print('Maximum Value:', best_value)
print('Last DP Row:', dp_table[-1])
print('Time Complexity  : O(nW)')
print('Space Complexity : O(nW)')
print('Observation      : Dynamic Programming finds the global optimum for indivisible items.')

## Problem 5: Shortest Path Problem - Dijkstra's Algorithm

### Problem Context
Navigation systems and communication networks often need the shortest path from one source to all other destinations. Dijkstra's algorithm is a standard solution when all edge weights are non-negative.

### Why This Strategy Fits
The algorithm repeatedly selects the nearest unvisited node and relaxes its edges. This greedy graph strategy produces correct shortest distances for non-negative weights.

In [ ]:
def dijkstra(graph, source):
    distances = {node: float('inf') for node in graph}
    distances[source] = 0
    priority_queue = [(0, source)]

    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)

        if current_distance > distances[current_node]:
            continue

        for neighbor, weight in graph[current_node].items():
            distance = current_distance + weight
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(priority_queue, (distance, neighbor))

    return distances

graph = {
    'A': {'B': 4, 'C': 2},
    'B': {'A': 4, 'C': 1, 'D': 5},
    'C': {'A': 2, 'B': 1, 'D': 8, 'E': 10},
    'D': {'B': 5, 'C': 8, 'E': 2, 'F': 6},
    'E': {'C': 10, 'D': 2, 'F': 3},
    'F': {'D': 6, 'E': 3}
}

source_node = 'A'
shortest_distances = dijkstra(graph, source_node)

print('Source Node:', source_node)
print('Shortest Distances:')
for node, distance in shortest_distances.items():
    print(f'  {source_node} -> {node} = {distance}')

print('Time Complexity  : O((V + E) log V) with a priority queue')
print('Space Complexity : O(V)')
print("Observation      : Dijkstra's algorithm is useful for route planning and network routing.")

## Final Comparison and Reflection

| Strategy | Algorithm | Time Complexity | Best Use Case | Key Insight |
| --- | --- | --- | --- | --- |
| Divide and Conquer | Merge Sort | `O(n log n)` | Large-scale sorting | Balanced splitting improves scalability |
| Comparative Study | Bubble vs Merge Sort | `O(n^2)` vs `O(n log n)` | Performance evaluation | Practical results match theory |
| Greedy | Fractional Knapsack | `O(n log n)` | Divisible resource allocation | Local best choices lead to optimum |
| Dynamic Programming | 0/1 Knapsack | `O(nW)` | Indivisible optimization problems | Stored subproblems prevent recomputation |
| Graph Algorithm | Dijkstra's Algorithm | `O((V+E) log V)` | Routing and navigation | Nearest-node expansion works for non-negative edges |

### Reflection
The results show that different problems require different strategies. Merge Sort is more scalable than Bubble Sort, Greedy works well when local decisions remain globally valid, Dynamic Programming is necessary when greedy choices are not enough, and Dijkstra's algorithm is effective for shortest-path computation in weighted graphs.